In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import SymbolDataset, SymbolClassifier

In [ ]:
# Classifier Parameters
sf = 9
input = 256
hidden = 1024
output = 2 ** sf
lr=5e-3
batch_size=32

folder_path = "classifier_dataset_sf{}_{}_{}".format(sf, input, output)

X = np.load(f"{folder_path}/X.npy")   # shape (30720, 16)
y = np.load(f"{folder_path}/y.npy")   # shape (30720,)

dataset = SymbolDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

layers = [input,hidden,output]
model = SymbolClassifier(layers).to(device)
criterion = nn.CrossEntropyLoss()   # Softmax included
optimizer = optim.Adam(model.parameters(), lr=lr,weight_decay=1e-4) #weight_decay=1e-4

In [3]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)              # (batch, 512)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    acc = correct / total * 100

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")
    
torch.save(model.state_dict(), "symbol_classifier.pt")

Epoch [1/30] Loss: 3.2056 | Accuracy: 14.00%
Epoch [2/30] Loss: 2.5345 | Accuracy: 39.00%
Epoch [3/30] Loss: 2.1859 | Accuracy: 41.62%
Epoch [4/30] Loss: 2.0367 | Accuracy: 43.13%
Epoch [5/30] Loss: 1.9180 | Accuracy: 45.63%
Epoch [6/30] Loss: 1.8314 | Accuracy: 46.95%
Epoch [7/30] Loss: 1.7462 | Accuracy: 48.26%
Epoch [8/30] Loss: 1.6821 | Accuracy: 49.77%
Epoch [9/30] Loss: 1.6262 | Accuracy: 49.99%
Epoch [10/30] Loss: 1.5722 | Accuracy: 51.78%
Epoch [11/30] Loss: 1.5279 | Accuracy: 52.42%
Epoch [12/30] Loss: 1.4935 | Accuracy: 53.64%
Epoch [13/30] Loss: 1.4500 | Accuracy: 54.03%
Epoch [14/30] Loss: 1.4142 | Accuracy: 54.95%
Epoch [15/30] Loss: 1.3828 | Accuracy: 55.51%
Epoch [16/30] Loss: 1.3659 | Accuracy: 56.29%
Epoch [17/30] Loss: 1.3212 | Accuracy: 57.39%
Epoch [18/30] Loss: 1.2930 | Accuracy: 57.99%
Epoch [19/30] Loss: 1.2618 | Accuracy: 58.89%
Epoch [20/30] Loss: 1.2382 | Accuracy: 59.53%
Epoch [21/30] Loss: 1.2073 | Accuracy: 60.33%
Epoch [22/30] Loss: 1.1888 | Accuracy: 60.6

In [4]:

def evaluate(model, dataloader):
    model.eval()
    correct1 = 0
    correct2 = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            _, top2 = torch.topk(logits, k=2, dim=1)
            preds = torch.argmax(logits, dim=1)

            correct1 += (preds == y_batch).sum().item()
            correct2 += (top2 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total += y_batch.size(0)

    print(f"Top-1 Accuracy: {100*correct1/total:.2f}%")
    print(f"Top-2 Accuracy: {100*correct2/total:.2f}%")


# Load

model.load_state_dict(torch.load("symbol_classifier.pt"))
model.eval()
evaluate(model,dataloader)

Top-1 Accuracy: 40.00%
Top-2 Accuracy: 66.09%
